# GRU attention decoder: neural dynamics to DINOv3 layer features

## Main idea

This notebook reverses `GRU_attn_model_dev.ipynb`. Its input is the complete AIT time series and its targets are train-only PCA reductions of one shallow, one middle, and one deep DINOv3 layer. Repeated presentations of an image remain in one split. PCA and neural-channel normalization are fitted on training images/trials only.

```text
Neural response X [B, T, C]
    |
    +-- optional Linear(C -> C_b) + LayerNorm + dropout
    |
    v
Variational-dropout GRU -> all states H [B, T, H]
    |
    +-- q_shallow -> softmax over T -> z_shallow -> head -> PCA shallow
    +-- q_mid     -> softmax over T -> z_mid     -> head -> PCA mid
    +-- q_deep    -> softmax over T -> z_deep    -> head -> PCA deep
```

For target layer $l$, the independent learned query $q_l$ gives

$$s_{blt}=q_l^T h_{bt}/\sqrt{H},\quad \alpha_{bl}=\operatorname{softmax}_t(s_{bl}),\quad z_{bl}=\sum_t\alpha_{blt}h_{bt}.$$

The three queries share the GRU sequence but not a pooled vector or regression head. The decoder is deterministic in evaluation mode, so it estimates conditional means and remains compatible with later noise-ceiling-fraction analysis. Attention is descriptive: it shows where the fitted decoder pools information, not causal importance of a neural time bin.

In [ ]:
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

# Locate the repository whether Jupyter starts at root or in scripts/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [cwd, *cwd.parents] if (path / 'config.yaml').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate config.yaml.')
# end if project root was not found

ENV = os.getenv('MY_ENV', 'tiziano_mac_mini')
with open(PROJECT_ROOT / 'config.yaml', 'r') as f:
    config = yaml.safe_load(f)
paths = config[ENV]['paths']

project_src_path = str((PROJECT_ROOT / 'python_scripts' / 'src').resolve())
useful_stuff_path = str(Path(paths['useful_stuff_path']).resolve())
for source_path in (useful_stuff_path, project_src_path):
    while source_path in sys.path:
        sys.path.remove(source_path)
    # end while source path is already registered
    sys.path.insert(0, source_path)
# end for source path

from IT_recap.hf_feature_extraction import is_valid_image_file, load_hf_layer_features
from IT_recap.neural_feature_decoding import (
    NeuralLayerFeatureDataset,
    average_rows_by_image,
    collect_feature_decoder_outputs,
    fit_feature_decoder,
    fit_layer_pcas,
    fit_neural_channel_standardizer,
    layer_balanced_mse_loss,
    plot_layer_feature_reconstruction,
)
from model_classes.temporal_models import NeuralToLayerFeatureGRUModel
from project_specific_utils.dataloader import (
    load_img_natraster,
    load_img_raster,
    map_image_order_from_ann_to_monkey,
    map_trial_image_order_to_ann,
)

In [ ]:
@dataclass
class Cfg:
    # Neural recording and image alignment.
    neural_data_source: str = 'natraster'  # 'raster' (~22K trials) or 'natraster'
    monkey_name: str = 'three0'
    date: str = '250313'
    brain_area: str = 'AIT'
    folder_name: str = 'talia_20each_tizi'
    raster_file: str = 'rasters_three0_250313to21.mat'
    raster_key: str = 'rasters'
    raster_image_names_file: str = 'allimages_three0_250313to21.mat'
    raster_image_names_key: str = 'allimages'
    original_fs: int = 1000
    new_fs: int = 100
    time_start_ms: float = 0.0
    time_end_ms: float = 300.0
    raster_chunk_size: int = 512

    # Three DINOv3 targets, independently reduced with train-only PCA.
    model_name: str = 'dino_v3_l'
    img_size: int = 224
    pooling: str = 'mean'
    layer_names: list[str] = field(default_factory=lambda: [
        'layer.3.mlp.down_proj',
        'layer.13.mlp.down_proj',
        'layer.20.mlp.down_proj',
    ])
    layer_labels: list[str] = field(default_factory=lambda: [
        'shallow', 'mid', 'deep'
    ])
    target_pca_components: int = 64
    whiten_pca: bool = True

    # Image-grouped split and optimization.
    validation_fraction: float = 0.2
    random_seed: int = 0
    batch_size: int = 256
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4

    # Manually selected architecture. Edit these values directly.
    hidden_dim: int = 16  # Keep in the small {8, 16, 32} range.
    n_gru_layers: int = 1
    bottleneck_dim: int | None = 16  # Set to None to bypass it.
    head_type: str = 'linear'  # 'linear' or 'mlp'
    head_hidden_dim: int = 16  # Used only by the MLP head.
    variational_dropout: float = 0.15
    hidden_dropout: float = 0.25
    head_dropout: float = 0.25

    epochs: int = 200
    patience: int = 20
    reconstruction_sample_index: int = 10
    ridge_alphas: np.ndarray = field(
        default_factory=lambda: np.logspace(-4, 4, 17)
    )
# EOC

cfg = Cfg()
cfg

## Load and align neural inputs with static DINOv3 targets

The cached feature array remains in `ImageFolder` order. Neural samples are mapped to that order exactly as in the forward notebook. The model input is transposed to `(presentation, time, channels)` only after alignment.

In [ ]:
dataset_path = Path(paths['livingstone_lab']) / 'Stimuli' / cfg.folder_name
image_dataset = ImageFolder(
    root=dataset_path,
    is_valid_file=is_valid_image_file,
    allow_empty=True,
)

if cfg.neural_data_source == 'raster':
    raster, neural_image_names = load_img_raster(
        paths=paths,
        monkey_name=cfg.monkey_name,
        raster_file=cfg.raster_file,
        image_names_file=cfg.raster_image_names_file,
        raster_key=cfg.raster_key,
        image_names_key=cfg.raster_image_names_key,
        original_fs=cfg.original_fs,
        new_fs=cfg.new_fs,
        time_start_ms=cfg.time_start_ms,
        time_end_ms=cfg.time_end_ms,
        brain_area=cfg.brain_area,
        chunk_size=cfg.raster_chunk_size,
    )
    neural_image_indices = map_trial_image_order_to_ann(
        neural_image_names, image_dataset
    )
elif cfg.neural_data_source == 'natraster':
    raster = load_img_natraster(
        paths=paths,
        monkey_name=cfg.monkey_name,
        date=cfg.date,
        original_fs=cfg.original_fs,
        new_fs=cfg.new_fs,
        time_start_ms=cfg.time_start_ms,
        time_end_ms=cfg.time_end_ms,
        brain_area=cfg.brain_area,
    )
    neural_image_indices = np.asarray(
        map_image_order_from_ann_to_monkey(
            paths, cfg.monkey_name, cfg.date, image_dataset
        ),
        dtype=int,
    )
else:
    raise ValueError("neural_data_source must be 'raster' or 'natraster'.")
# end if repeated or averaged neural data are requested

raster_array = raster.get_array()
layer_features = load_hf_layer_features(
    output_dir=Path(paths['data_path']) / 'models',
    dataset_name=cfg.folder_name,
    model_name=cfg.model_name,
    img_size=cfg.img_size,
    layer_names=cfg.layer_names,
    pooling=cfg.pooling,
)
if layer_features.shape[:2] != (len(image_dataset), len(cfg.layer_names)):
    raise ValueError('Cached targets do not align with images and layer names.')
# end if cached target alignment is invalid
print(f'neural [channels, time, samples]: {raster_array.shape}')
print(f'DINOv3 [images, layers, embedding]: {layer_features.shape}')
print(f'aligned unique images: {len(np.unique(neural_image_indices))}')

## Leakage-safe image split, PCA targets, and neural scaling

Unique images are split before trials, so repetitions never cross partitions. Each layer gets its own 64-component PCA. Whitening is fitted only on training images and gives retained target coordinates comparable scale; the loss then averages the three layer MSEs equally. Neural channels are standardized from training trials only.

In [ ]:
if not 0.0 < cfg.validation_fraction < 1.0:
    raise ValueError('validation_fraction must lie between zero and one.')
# end if validation fraction is invalid
split_rng = np.random.default_rng(cfg.random_seed)
unique_image_indices = np.unique(neural_image_indices)
shuffled_images = split_rng.permutation(unique_image_indices)
n_validation_images = max(1, round(len(shuffled_images) * cfg.validation_fraction))
validation_image_indices = shuffled_images[:n_validation_images]
training_image_indices = shuffled_images[n_validation_images:]
is_validation_trial = np.isin(neural_image_indices, validation_image_indices)
training_trial_indices = np.flatnonzero(~is_validation_trial)
validation_trial_indices = np.flatnonzero(is_validation_trial)
if set(training_image_indices) & set(validation_image_indices):
    raise RuntimeError('An image appears in both training and validation.')
# end if repeated stimuli leaked across the split

image_targets, layer_pcas, target_slices = fit_layer_pcas(
    layer_features,
    training_image_indices,
    n_components=cfg.target_pca_components,
    random_seed=cfg.random_seed,
    whiten=cfg.whiten_pca,
)
target_dims = [target_slice.stop - target_slice.start for target_slice in target_slices]
channel_mean, channel_std = fit_neural_channel_standardizer(
    raster_array, training_trial_indices
)
full_dataset = NeuralLayerFeatureDataset(
    raster_array,
    image_targets,
    neural_image_indices,
    channel_mean,
    channel_std,
)
training_dataset = Subset(full_dataset, training_trial_indices.tolist())
validation_dataset = Subset(full_dataset, validation_trial_indices.tolist())
pin_memory = torch.cuda.is_available()

def make_loaders(seed):
    # Re-seeding makes repeated manual runs use the same minibatch order.
    generator = torch.Generator().manual_seed(seed)
    training_loader = DataLoader(
        training_dataset, batch_size=cfg.batch_size, shuffle=True,
        pin_memory=pin_memory, generator=generator,
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=cfg.batch_size, shuffle=False,
        pin_memory=pin_memory,
    )
    return training_loader, validation_loader
# EOF

for label, pca in zip(cfg.layer_labels, layer_pcas):
    explained = pca.explained_variance_ratio_.sum()
    print(f'{label:8s}: {pca.n_components_} PCs, {explained:.3f} variance retained')
# end for fitted layer PCA
print(
    f'split: {len(training_image_indices)} images / {len(training_trial_indices)} trials train; '
    f'{len(validation_image_indices)} images / {len(validation_trial_indices)} trials validation'
)

## Architecture inspection

The architecture is controlled directly by `Cfg`; there is no automatic search. The next cell constructs the model, prints the complete PyTorch module tree and trainable parameter counts, then runs one validation batch to display every important tensor shape.

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
# end if an accelerator is available

torch.manual_seed(cfg.random_seed)
model = NeuralToLayerFeatureGRUModel(
    n_neural_channels=raster_array.shape[0],
    target_dims=target_dims,
    target_names=cfg.layer_labels,
    hidden_dim=cfg.hidden_dim,
    n_gru_layers=cfg.n_gru_layers,
    bottleneck_dim=cfg.bottleneck_dim,
    variational_dropout=cfg.variational_dropout,
    hidden_dropout=cfg.hidden_dropout,
    head_dropout=cfg.head_dropout,
    head_type=cfg.head_type,
    head_hidden_dim=cfg.head_hidden_dim,
).to(device)
training_loader, validation_loader = make_loaders(cfg.random_seed)

# Print the actual modules and how capacity is distributed across them.
print(model)
parameter_counts = {}
for parameter_name, parameter in model.named_parameters():
    if parameter.requires_grad:
        block_name = parameter_name.split('.')[0]
        parameter_counts[block_name] = (
            parameter_counts.get(block_name, 0) + parameter.numel()
        )
    # end if parameter is trainable
# end for model parameter
print('\nTrainable parameters by block:')
for block_name, count in parameter_counts.items():
    print(f'  {block_name:28s} {count:8,d}')
# end for model block
print(f"  {'total':28s} {sum(parameter_counts.values()):8,d}")

# Verify that all GRU states reach three independent temporal attention rows.
example_neural, example_targets = next(iter(validation_loader))
model.eval()
with torch.no_grad():
    example_predictions, example_attention, example_hidden = model(
        example_neural.to(device), return_hidden_states=True
    )
# end with no gradient tracking
print('\nShape trace:')
print(f'  neural input      {tuple(example_neural.shape)}  [B, T, C]')
print(f'  all GRU states   {tuple(example_hidden.shape)}  [B, T, H]')
print(f'  layer attention  {tuple(example_attention.shape)}  [B, layers, T]')
print(f'  PCA prediction   {tuple(example_predictions.shape)}  [B, sum(D_l)]')
print(f'  PCA target       {tuple(example_targets.shape)}  [B, sum(D_l)]')

## Training and flattened-neural ridge reference

The manually configured architecture is trained with validation-based early stopping. Ridge receives the same standardized neural time series flattened over time and channels and predicts the same whitened PCA targets on the same split. It is a useful non-recurrent reference, although its parameter count is much larger than the small GRU.

In [ ]:
final_history = fit_feature_decoder(
    model,
    training_loader,
    validation_loader,
    target_slices,
    learning_rate=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    max_epochs=cfg.epochs,
    patience=cfg.patience,
    device=device,
)

all_neural_inputs = full_dataset.neural_sequences.numpy().reshape(len(full_dataset), -1)
all_targets = full_dataset.image_targets[full_dataset.image_indices].numpy()
ridge = RidgeCV(alphas=cfg.ridge_alphas)
ridge.fit(all_neural_inputs[training_trial_indices], all_targets[training_trial_indices])
ridge_validation_predictions = ridge.predict(all_neural_inputs[validation_trial_indices])
ridge_validation_targets = all_targets[validation_trial_indices]
ridge_layer_mse = [
    np.mean((ridge_validation_predictions[:, s] - ridge_validation_targets[:, s]) ** 2)
    for s in target_slices
]
ridge_validation_mse = float(np.mean(ridge_layer_mse))
print(
    f"final GRU best validation MSE {final_history['best_validation_loss']:.5f} "
    f"at epoch {final_history['best_epoch']} | ridge {ridge_validation_mse:.5f} "
    f"(alpha={ridge.alpha_:.3g})"
)

## Training curves, ridge reference, and per-layer neural-time attention

The attention panel is the interpretability-critical reversal of the original notebook: rows are DINOv3 target depth and columns are neural time. Every row sums to one independently.

In [ ]:
validation_predictions, validation_targets, validation_attention, validation_hidden = (
    collect_feature_decoder_outputs(model, validation_loader, device=device)
)
image_attention, _ = average_rows_by_image(
    validation_attention, neural_image_indices[validation_trial_indices]
)
mean_attention = image_attention.mean(axis=0)
time_ms = cfg.time_start_ms + np.arange(mean_attention.shape[1]) * 1000.0 / cfg.new_fs
training_epochs = np.arange(1, len(final_history['training_losses']) + 1)
validation_epochs = np.arange(len(final_history['validation_losses']))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(training_epochs, final_history['training_losses'], label='Training MSE')
axes[0].plot(validation_epochs, final_history['validation_losses'], label='Validation MSE')
axes[0].scatter(
    final_history['best_epoch'], final_history['best_validation_loss'],
    marker='D', s=70, zorder=4, label='Best GRU validation MSE',
)
axes[0].scatter(
    validation_epochs[-1], ridge_validation_mse, marker='*', s=180,
    color='black', zorder=4, label='Flattened-neural ridge MSE',
)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Layer-balanced whitened PCA MSE')
axes[0].set_title('DINOv3 feature prediction error')
axes[0].grid(alpha=0.3)
axes[0].legend()

attention_image = axes[1].imshow(
    mean_attention, aspect='auto', interpolation='nearest', cmap='viridis',
    extent=[time_ms[0], time_ms[-1], len(cfg.layer_labels) - 0.5, -0.5],
)
axes[1].set_xlabel('Neural time (ms)')
axes[1].set_ylabel('Predicted DINOv3 layer')
axes[1].set_title('Mean validation attention over neural time')
axes[1].set_yticks(np.arange(len(cfg.layer_labels)))
axes[1].set_yticklabels(cfg.layer_labels)
fig.colorbar(attention_image, ax=axes[1], label='Attention probability')
fig.tight_layout()
plt.show()

sample_idx = min(cfg.reconstruction_sample_index, len(validation_predictions) - 1)
plot_layer_feature_reconstruction(
    validation_predictions[sample_idx],
    validation_targets[sample_idx],
    target_slices,
    cfg.layer_labels,
    title=f'Validation presentation {sample_idx}: PCA target reconstruction',
)

## Image-level feature accuracy and representational geometry

Repeated validation presentations are averaged by image before scoring. This prevents images with more repetitions from dominating the summary. $R^2$ and flattened prediction correlation quantify coordinate recovery; RDM Spearman correlation asks whether the predicted layer preserves the target image geometry.

In [ ]:
validation_trial_image_indices = neural_image_indices[validation_trial_indices]
image_predictions, scored_image_indices = average_rows_by_image(
    validation_predictions, validation_trial_image_indices
)
image_targets_scored, target_image_indices = average_rows_by_image(
    validation_targets, validation_trial_image_indices
)
if not np.array_equal(scored_image_indices, target_image_indices):
    raise RuntimeError('Prediction and target image orders differ.')
# end if image-level values are misaligned

fig, axes = plt.subplots(2, len(target_slices), figsize=(5 * len(target_slices), 8))
for layer_idx, (layer_label, target_slice) in enumerate(
    zip(cfg.layer_labels, target_slices)
):
    predicted_layer = image_predictions[:, target_slice]
    target_layer = image_targets_scored[:, target_slice]
    layer_r2 = r2_score(target_layer, predicted_layer, multioutput='variance_weighted')
    feature_r = np.corrcoef(target_layer.ravel(), predicted_layer.ravel())[0, 1]
    target_rdm = squareform(pdist(target_layer, metric='correlation'))
    predicted_rdm = squareform(pdist(predicted_layer, metric='correlation'))
    upper_triangle = np.triu_indices_from(target_rdm, k=1)
    geometry_r = spearmanr(
        target_rdm[upper_triangle], predicted_rdm[upper_triangle]
    ).statistic
    print(
        f'{layer_label:8s} | image-level R2 {layer_r2:.3f} | '
        f'feature r {feature_r:.3f} | RDM rho {geometry_r:.3f}'
    )

    value_limit = np.nanmax(np.abs([target_rdm, predicted_rdm]))
    target_image = axes[0, layer_idx].imshow(
        target_rdm, cmap='viridis', vmin=0.0, vmax=value_limit
    )
    axes[0, layer_idx].set_title(f'{layer_label}: target PCA RDM')
    axes[1, layer_idx].imshow(
        predicted_rdm, cmap='viridis', vmin=0.0, vmax=value_limit
    )
    axes[1, layer_idx].set_title(f'{layer_label}: decoded PCA RDM\nρ={geometry_r:.3f}')
    fig.colorbar(target_image, ax=axes[:, layer_idx], fraction=0.025)
# end for target layer
for axis in axes.ravel():
    axis.set_xlabel('Validation image')
    axis.set_ylabel('Validation image')
# end for geometry panel
plt.show()